# Download Places365 with Torchvision

This notebook downloads the **Places365-Standard** dataset using `torchvision.datasets.Places365` with `download=True`.

For DreamVision 2.0, we want:
- `split="train-standard"`
- `small=True` for the `256x256` images
- optional `val` download for sanity checks or later evaluation


## Why This Approach

Using `torchvision` keeps the download path inside the PyTorch workflow, so we do not need TensorFlow or manual archive handling for the first pass.

The local `torchvision` implementation supports these splits:
- `train-standard`
- `train-challenge`
- `val`
- `test`

We are intentionally choosing **Standard**, not **Challenge**, because it is much lighter and already large enough for background GAN training.


In [1]:
from __future__ import annotations

from pathlib import Path

import torchvision
from torchvision.datasets import Places365

print("torchvision:", torchvision.__version__)


torchvision: 0.20.1


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_ROOT = PROJECT_ROOT / "data" / "places365_torchvision"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_SPLIT = "train-standard"
DOWNLOAD_SMALL_IMAGES = True
DOWNLOAD_VAL_SPLIT = True

print("Dataset root:", DATA_ROOT)
print("Train split:", TRAIN_SPLIT)
print("small:", DOWNLOAD_SMALL_IMAGES)
print("download val:", DOWNLOAD_VAL_SPLIT)


Dataset root: /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision
Train split: train-standard
small: True
download val: True


## What Torchvision Downloads

With `split="train-standard"` and `small=True`, `torchvision` downloads the official MIT archive:
- `train_256_places365standard.tar`

With `split="val"` and `small=True`, it downloads:
- `val_256.tar`

Those files come from the official Places365 MIT host under `http://data.csail.mit.edu/places/places365/`.


## SSL Workaround

Some cluster environments fail on the Places365 metadata download with an SSL EOF error even when the dataset call itself is correct.

If that happens, run the next cell before calling `Places365(..., download=True)`.


In [3]:
import ssl

# Some HPC environments break on redirected HTTPS requests during dataset download.
# This relaxes certificate verification for the current Python process only.
ssl._create_default_https_context = ssl._create_unverified_context

print("SSL workaround enabled for this notebook session.")


SSL workaround enabled for this notebook session.


## Fallback If `download=True` Still Fails

If the cluster still blocks the automatic download, use `wget` to fetch the same official MIT files, then keep using `torchvision.datasets.Places365` with `download=False`.


In [4]:
TRAIN_URL = "http://data.csail.mit.edu/places/places365/train_256_places365standard.tar"
VAL_URL = "http://data.csail.mit.edu/places/places365/val_256.tar"
DEVKIT_URL = "http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar"

print("If needed, run these in a notebook cell with ! prefix:")
print(f"wget -c --no-check-certificate -P {DATA_ROOT} {DEVKIT_URL}")
print(f"wget -c --no-check-certificate -P {DATA_ROOT} {TRAIN_URL}")
print(f"wget -c --no-check-certificate -P {DATA_ROOT} {VAL_URL}")
print(f"tar -xf {DATA_ROOT / "filelist_places365-standard.tar"} -C {DATA_ROOT}")
print(f"tar -xf {DATA_ROOT / "train_256_places365standard.tar"} -C {DATA_ROOT}")
print(f"tar -xf {DATA_ROOT / "val_256.tar"} -C {DATA_ROOT}")


SyntaxError: f-string: expecting '}' (2035413906.py, line 9)

In [5]:
train_dataset = Places365(
    root=str(DATA_ROOT),
    split=TRAIN_SPLIT,
    small=DOWNLOAD_SMALL_IMAGES,
    download=True,
)

print("Train download complete.")
print("If this cell raises an SSL EOF error, run the SSL workaround cell above and retry.")
print("Number of train samples:", len(train_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Images directory:", train_dataset.images_dir)


100%|██████████████████████████████████████| 67.5M/67.5M [00:16<00:00, 4.00MB/s]


Extracting /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision/filelist_places365-standard.tar to /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision


100%|████████████████████████████████████| 26.1G/26.1G [1:29:46<00:00, 4.85MB/s]


Extracting /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision/train_256_places365standard.tar to /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision
Train download complete.
If this cell raises an SSL EOF error, run the SSL workaround cell above and retry.
Number of train samples: 1803460
Number of classes: 365
Images directory: /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision/data_256_standard


In [6]:
val_dataset = None
if DOWNLOAD_VAL_SPLIT:
    val_dataset = Places365(
        root=str(DATA_ROOT),
        split="val",
        small=DOWNLOAD_SMALL_IMAGES,
        download=True,
    )
    print("Validation download complete.")
    print("Number of val samples:", len(val_dataset))
    print("Validation images directory:", val_dataset.images_dir)
else:
    print("Skipping validation download.")


100%|████████████████████████████████████████| 525M/525M [01:19<00:00, 6.59MB/s]


Extracting /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision/val_256.tar to /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision
Validation download complete.
Number of val samples: 36500
Validation images directory: /Users/shrimayee/sdeshpan/ADL/DreamTales/DreamVision 2.0/data/places365_torchvision/val_256


In [7]:
print("Top-level contents:")
for child in sorted(DATA_ROOT.iterdir()):
    print("-", child.name)


Top-level contents:
- .DS_Store
- categories_places365.txt
- data_256_standard
- filelist_places365-standard.tar
- places365_test.txt
- places365_train_standard.txt
- places365_val.txt
- train_256_places365standard.tar
- val_256
- val_256.tar


## Important Note for Training

`torchvision.datasets.Places365` can read the official archive layout directly, but it does **not** behave like a simple `ImageFolder` tree.

That means for the background GAN notebook, the cleanest next step is to load Places365 using `Places365(...)` directly instead of assuming `train/<class_name>/*.jpg`.

This will save us from manual folder reorganization.


In [8]:
sample_image, sample_target = train_dataset[0]
print("First sample target:", sample_target)
print("First sample size:", getattr(sample_image, "size", None))


First sample target: 0
First sample size: (256, 256)


## Next Step

After this notebook succeeds on HiPerGator, the next move should be updating the background GAN training notebook to use `torchvision.datasets.Places365` directly for its dataloader.
